# 개별종목 조합H — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합H 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합H의 피처 값만 지정합니다.
import json

COMBINATION = 'H'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_regime',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합H 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_regime', 'turnover_20')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.3882,0.5012,-0.1130,0.3451,0.2489,0.3161
1,2,balanced,980,20150123,20150421,0.3741,0.3978,-0.0237,0.3585,0.3012,0.3416
2,3,balanced,1210,20151228,20160328,0.3484,0.3762,-0.0277,0.3425,0.3157,0.3349
3,4,balanced,1439,20161202,20170228,0.4059,0.4617,-0.0558,0.3763,0.2798,0.3451
4,5,balanced,1669,20171113,20180207,0.3732,0.3901,-0.0169,0.3623,0.3603,0.3652
5,6,balanced,1899,20181024,20190118,0.3890,0.3725,0.0166,0.3870,0.3914,0.3891
6,7,balanced,2129,20190930,20191224,0.3900,0.4781,-0.0881,0.3546,0.3353,0.3586
7,8,balanced,2359,20200902,20201130,0.3459,0.3476,-0.0017,0.3456,0.4233,0.3682
8,9,balanced,2589,20210806,20211105,0.3539,0.3916,-0.0377,0.3415,0.3738,0.3559
9,10,balanced,2818,20220714,20221012,0.3658,0.3454,0.0204,0.3643,0.3373,0.3553


,OOS 폴드 평균
accuracy,0.3727
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0242
macro_f1,0.3590
down_recall,0.3470
core_harmonic_mean,0.3570


재실행 명령: python scripts/run_stock_model_experiment.py
